# Stage 7 — Live Data Pipeline (Adzuna API)

**Goal:** check whether the SkillMap findings hold on *current* job postings. Live US IT postings come from the Adzuna API. They go through the Stage 2 skill extraction and the exact training salary thresholds, get loaded into the warehouse, and are tested against the Stage 4 rules and the Stage 5 model.

**API budget:** the free tier allows **25 calls in total**. `src/live_pipeline.py` records every request in `data/raw/adzuna_live/api_calls.json` before sending it, refuses to call past 25, and caches the health check and every page. **This notebook makes no API calls.** It reads the cached data fetched on 2026-10-14.

**Credentials:** read from the gitignored `.env` (`ADZUNA_APP_ID`, `ADZUNA_API_KEY`; see `.env.example`). Adzuna authenticates with `app_id` + `app_key`; a key-only request returned HTTP 400. Adzuna embeds the app ID in each `redirect_url`, so cached pages are scrubbed before use.

The logic lives in `src/live_pipeline.py`. To run without the notebook: `python -m src.live_pipeline`.

## Setup

In [1]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd


def find_root(start: Path) -> Path:
    """Return the first directory at or above `start` that contains CLAUDE.md."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (no CLAUDE.md found above cwd).")


PROJECT_ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))

from src import live_pipeline as lp
from src.preprocessing import load_cleaned_jobs

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
OUTPUT_DIR = PROJECT_ROOT / "outputs"
paths = lp.live_paths(PROJECT_ROOT)
ledger = lp.read_ledger(paths["ledger"])
print(f"API calls recorded: {len(ledger)} of {lp.CALL_BUDGET}")
pd.DataFrame(ledger)

API calls recorded: 25 of 25


,call,time,page,results_per_page,status
0,1,2026-10-14T12:31:52,1,10,400
1,2,2026-10-14T12:36:03,1,10,200
2,3,2026-10-14T12:36:09,1,50,200
3,4,2026-10-14T12:36:11,2,50,200
4,5,2026-10-14T12:36:13,3,50,200
5,6,2026-10-14T12:36:15,4,50,200
6,7,2026-10-14T12:36:19,5,50,200
7,8,2026-10-14T12:36:21,6,50,200
8,9,2026-10-14T12:36:23,7,50,200
9,10,2026-10-14T12:36:25,8,50,200


## Step 1 — API health check (1 call)

One call to page 1 with `results_per_page=10` and `category=it-jobs`. The fetch only proceeds on HTTP 200 with results.

The ledger above shows two health checks. The first sent only `api_key` and got **HTTP 400**. The second, with `app_id` + `app_key`, returned **HTTP 200**. The cached result of the successful check is shown here.

In [2]:
health = lp.health_check(paths)
print(f"HTTP status: {health['status']}")
print(f"First result title: {health['first_title']!r}")
print(f"Adzuna total matching IT jobs: {health['count']:,}")
print("API responding — proceeding with full fetch" if health["ok"] else f"STOP: {health['error']}")

INFO src.live_pipeline: Health check cached from 2026-10-14T12:36:03: HTTP 200


HTTP status: 200
First result title: 'Associate Director, Technology Sourcing'
Adzuna total matching IT jobs: 488,952
API responding — proceeding with full fetch


## Step 2 — Fetch postings (remaining calls)

Pages of `results_per_page=50`, 1 second apart. The failed first health check used one call, so 23 calls were left for pages 1-23 (up to 1,150 postings) instead of the planned 24. `fetch_pages` skips cached pages and stops when the budget is exhausted, so this cell makes no calls.

Extracted fields: id, title, company and location display names, salary_min, salary_max, description, created, and redirect_url. Also kept: `state` (from Adzuna's structured `location.area`) and `salary_is_predicted` (whether Adzuna estimated the salary).

In [3]:
fetch = lp.fetch_pages(paths, health)
postings = lp.combine_pages(paths)
raw = lp.extract_postings(postings)
raw.to_csv(paths["raw_csv"], index=False)
counts = lp.salary_counts(raw)
print(fetch)
print(counts)
raw.drop(columns=["description", "redirect_url"]).head()

INFO src.live_pipeline: Fetch: 0 new calls, 23 pages cached (API call budget exhausted)


INFO src.live_pipeline: Saved 1150 unique postings to /Users/genjishimada49/Documents/CU-Boulder/Fall 2026/Data Mining/SkillMap/data/raw/adzuna_live/raw_jobs.json


{'pages_fetched': 0, 'pages_cached': 23, 'stopped_reason': 'API call budget exhausted'}
{'total': 1150, 'with_salary_min': 1150, 'with_salary_max': 1149, 'with_both': 1149, 'predicted_by_adzuna': 1092}


,id,title,company,location,state,salary_min,salary_max,created,salary_is_predicted
0,5919470903,"Associate Director, Technology Sourcing",CSL,"King of Prussia, Montgomery County",PA,164985.17,164985.17,2026-10-08T13:19:11Z,1
1,5864552127,"Epic Systems Analyst - HIM (ROI, DT, Coding & ...",OhioHealth,"Grandview, Franklin County",OH,121742.49,121742.49,2026-09-01T08:01:40Z,1
2,5921122923,Senior Project Manager,Maner Costerisan,"Lansing, Ingham County",MI,95000.00,125000.00,2026-10-09T15:13:32Z,0
3,5903001177,"Senior/Lead Data Scientist, Data & Analytics (...",Starbucks,"Brentwood, Williamson County",TN,102560.98,102560.98,2026-09-29T14:35:43Z,1
4,5926290288,"Sr Forward Deployed Engineer, AWS Forward Depl...","Amazon Web Services, Inc.","State Farm, Arlington County",VA,249359.52,249359.52,2026-10-13T19:20:54Z,1


Two facts shape every later step:
- **Descriptions are short snippets.** Adzuna returns only about the first 500 characters, so far fewer skills can be matched than in full LinkedIn descriptions.
- **Most salaries are Adzuna estimates.** Nearly every posting has a salary, but most have `salary_is_predicted = 1`: Adzuna's own model output, not advertised pay.

In [4]:
print(f"Mean description length: {raw['description'].fillna('').str.len().mean():.0f} characters")
pd.crosstab(raw["salary_min"].notna() & raw["salary_max"].notna(),
            pd.to_numeric(raw["salary_is_predicted"]).fillna(0).astype(int),
            rownames=["has both salary_min and salary_max"], colnames=["salary_is_predicted"])

Mean description length: 498 characters


salary_is_predicted,0,1
has both salary_min and salary_max,,
False,1,0
True,57,1092


## Step 3 — Preprocess live data

This uses the same steps as Stage 2:
- **Skills:** the same regex matcher (`src.preprocessing.match_description_skills`) and the same 245-skill vocabulary, with nested phrases matched first and context rules for ambiguous terms.
- **Salary:** `mean_salary` = (salary_min + salary_max) / 2 where both exist, and `has_salary` flags those rows.
- **Titles:** lowercase, whitespace stripped.
- **Salary tier:** the **exact training thresholds**, Low ≤ $62,400, Mid ≤ $109,200, High above.

In [5]:
fetch_date = ledger[-1]["time"][:10]
live = lp.preprocess_live(raw, lp.load_vocabulary(PROJECT_ROOT), fetch_date)
live.to_csv(paths["cleaned"], index=False)
live = lp.load_live_cleaned(paths["cleaned"]).merge(raw[["id", "created"]].astype({"id": str}), on="id", how="left")
live["description_length"] = raw["description"].fillna("").str.len().to_numpy()
train = load_cleaned_jobs(PROJECT_ROOT / "data" / "processed" / "cleaned_jobs.csv")
print(f"{len(live):,} postings; mean {live['matched_skills'].str.len().mean():.1f} skills "
      f"(training {train['matched_skills'].str.len().mean():.1f}); "
      f"{(live['matched_skills'].str.len() > 0).mean():.1%} with >= 1 skill")
pd.DataFrame({"live (with salary)": lp.tier_distribution(live["salary_tier"]),
              "training": lp.tier_distribution(train["salary_tier"])}).round(3)

INFO src.preprocessing: Matched skills: 51 nested phrases matched first (time management, project management, inventory management, written communication, verbal communication, team management, team leadership, food safety, risk management, patient education, business development, professional development, budget management, case management, performance management, ability to work in a fastpaced environment, regulatory compliance, continuing education, electrical engineering, financial reporting, selfmotivation, account management, strategic planning, civil engineering, change management, construction management, mechanical engineering, team collaboration, training and development, customer service skills, retail management, technical writing, rn license, time management skills, financial management, travel nursing, report writing, patient assessment, risk assessment, english proficiency, stress management, treatment planning, equipment maintenance, oral communication, ability to work 

INFO src.preprocessing: Matched description skills: 933 of 1150 jobs have at least one, mean 2.0 per job


1,150 postings; mean 2.0 skills (training 11.5); 81.1% with >= 1 skill


,live (with salary),training
salary_tier,,
Low,0.067,0.337
Mid,0.312,0.330
High,0.621,0.333


## Step 4 — Load into the warehouse

`live_job_postings` in `skillmap.db` has the same columns as the `job_postings` fact table, plus `source` ('adzuna_live'), `fetch_date`, and `has_salary`. It also uses the same grain: one row per (job, skill), and one row with `skill_id` NULL for jobs with no skill.
- **Location:** matched to `dim_location` on (city, state) where possible.
- **Company:** matched to `dim_company` on name, with 0 = Unknown otherwise.
- **Date:** `time_id` is the posting date.

The table is dropped and recreated on each run. Note that rebuilding the warehouse with `python -m src.warehouse` recreates the database without it.

In [6]:
warehouse = lp.load_into_warehouse(live, paths["db"])
print(f"SELECT COUNT(*), SUM(has_salary) FROM live_job_postings -> "
      f"{warehouse['rows']:,}, {warehouse['rows_with_salary']:,}  (job x skill rows)")
warehouse

INFO src.live_pipeline: live_job_postings: 2570 rows (1150 jobs), 1149 jobs with salary


SELECT COUNT(*), SUM(has_salary) FROM live_job_postings -> 2,570, 2,569  (job x skill rows)


{'rows': 2570,
 'rows_with_salary': 2569,
 'jobs': 1150,
 'jobs_with_salary': 1149,
 'jobs_matched_location': 525,
 'jobs_matched_company': 307}

## Step 5 — Association rules on live data

Apriori runs on postings with at least one skill, with confidence > 0.5 and lift > 1.5, at `min_support = 0.05`. It retries at 0.02 if that gives fewer than 20 rules. The live rules are then compared with the Stage 4 top-20 training rules: in both, new in live, or disappeared in live. Disappeared rules also show their support, confidence, and lift recomputed on the live data.

In [7]:
onehot, rules, support = lp.mine_live_rules(live)
rules.to_csv(OUTPUT_DIR / "07_live_association_rules.csv", index=False)
comparison = lp.compare_rules(rules, pd.read_csv(OUTPUT_DIR / "04_association_rules.csv"), onehot)
comparison.to_csv(OUTPUT_DIR / "07_rules_comparison.csv", index=False)
print(f"min_support used: {support}; live rules: {len(rules)}")
print(comparison["status"].value_counts().to_string())
display(rules[["antecedents", "consequents", "support", "confidence", "lift"]])
comparison

INFO src.association: Apriori (min_support=0.05) on 933 transactions: 12 frequent itemsets


INFO src.association: Rules with confidence > 0.50 and lift > 1.50: 0


INFO src.live_pipeline: Only 0 rules at min_support=0.05; retrying at 0.02


INFO src.association: Apriori (min_support=0.02) on 933 transactions: 41 frequent itemsets


INFO src.association: Rules with confidence > 0.50 and lift > 1.50: 1


min_support used: 0.02; live rules: 1
status
disappeared_in_live    20
new_in_live             1


,antecedents,consequents,support,confidence,lift
0,compliance,security,0.0247,0.5349,2.6545


,status,antecedents,consequents,support_train,confidence_train,lift_train,support_live,confidence_live,lift_live
0,disappeared_in_live,accounting,finance,0.0546,0.5640,5.5405,0.0054,0.7143,41.6518
1,disappeared_in_live,finance,accounting,0.0546,0.5361,5.5405,0.0054,0.3125,41.6518
2,disappeared_in_live,marketing,sales,0.0700,0.5350,2.5168,0.0021,0.1429,19.0408
3,disappeared_in_live,"collaboration, management",development,0.0529,0.6444,1.7523,0.0011,1.0000,6.7609
4,disappeared_in_live,"communication, design",development,0.0581,0.6434,1.7496,0.0032,0.7500,5.0707
5,disappeared_in_live,"design, management",development,0.0746,0.6750,1.8355,0.0064,0.3158,2.1350
6,disappeared_in_live,"compliance, leadership",management,0.0561,0.7610,1.6851,0.0011,0.2500,1.9767
7,disappeared_in_live,design,development,0.1294,0.6191,1.6833,0.0504,0.2611,1.7653
8,disappeared_in_live,"design, engineering",development,0.0619,0.6604,1.7957,0.0107,0.2564,1.7336
9,disappeared_in_live,"development, engineering",design,0.0619,0.6320,3.0240,0.0107,0.2778,1.4398


## Step 6 — Salary-tier prediction on live data

The saved Stage 5 model (`05_best_model.pkl`) predicts a tier for every posting with a salary. Features are built with Stage 5's own functions and checked to match the training column order:
- the 245-skill multi-hot and the skill count
- the saved title and company TF-IDF vectorizers
- `experience_level` = Unknown, `company_size` = 0, industry = Other (Adzuna doesn't provide these)
- state from Adzuna's location

Accuracy is also reported separately for advertised salaries and for Adzuna-estimated ones.

In [8]:
preds, metrics = lp.predict_live(live, OUTPUT_DIR / "05_best_model.pkl")
if preds is None:
    print("Skipped:", metrics["skipped"])
else:
    preds.to_csv(OUTPUT_DIR / "07_live_predictions.csv", index=False)
    print(f"Accuracy {metrics['accuracy']:.3f}, macro F1 {metrics['f1_macro']:.3f}; "
          f"majority-class baseline {metrics['majority_baseline']:.3f}; "
          f"training test accuracy {metrics['train_accuracy']:.3f}")
    print(f"Advertised salaries: accuracy {metrics['accuracy_advertised']:.3f} (n={metrics['n_advertised']}); "
          f"Adzuna-estimated: {metrics['accuracy_adzuna_predicted']:.3f} (n={metrics['n_adzuna_predicted']})")
    display(metrics["confusion_matrix"])

INFO src.live_pipeline: Live predictions: 1149 postings, accuracy 0.594


Accuracy 0.594, macro F1 0.478; majority-class baseline 0.621; training test accuracy 0.729
Advertised salaries: accuracy 0.509 (n=57); Adzuna-estimated: 0.599 (n=1092)


,pred Low,pred Mid,pred High
actual Low,48,12,17
actual Mid,82,102,175
actual High,76,104,533


## Step 7 — Summary

In [9]:
summary = lp.build_summary(fetch_date, health, fetch, counts, live, train, warehouse, rules, support,
                           comparison, metrics, lp.read_ledger(paths["ledger"]))
(OUTPUT_DIR / "07_live_summary.txt").write_text(summary, encoding="utf-8")
print(summary)

SkillMap — Stage 7 Live Data Summary (Adzuna API)
Generated: 2026-10-14 12:39:16
Fetch date: 2026-10-14
Source: Adzuna US search, category=it-jobs. API calls used: 25 of 25: 2 health check(s) (HTTP 400, 200) + 23 pages of 50.

Fetch
Health check: HTTP 200, first title: 'Associate Director, Technology Sourcing', Adzuna total matching jobs: 488952
Pages cached: 23 (API call budget exhausted)
Total postings fetched (unique ids): 1,150
With salary_min: 1,150; with salary_max: 1,149; with both: 1,149 (99.9%)
Postings with salary (has_salary): 1,149 (99.9%); of these, Adzuna-estimated (salary_is_predicted=1): 1,092
State parsed for 89.8% of postings.
Skills: mean 2.0 per posting (training 11.5); 81.1% have >= 1 (training 99.3%). Adzuna descriptions are short snippets, mean 498 characters.

Top 10 skills in live data (share of postings) vs training
                live_share  train_share
matched_skills                         
engineering          0.170        0.170
security             0.163

## Findings

- **Fetch.** 25 of 25 calls were used: 2 health checks (HTTP 400 with `api_key` only, then 200 with `app_id` + `app_key`) and 23 pages, giving **1,150 unique postings**. 1,149 have a salary, but **95% of those (1,092) are Adzuna's estimates**. Only 57 are advertised salaries.
- **Pay.** The IT-only live sample is much better paid. With the training thresholds, 62% is High tier (vs 33% in training) and only 7% Low. The median is $123,689 vs $82,500.
- **Skills.** Snippets of about 500 characters give **2.0 skills per posting vs 11.5** in training, and 81% of postings have at least one skill (vs 99%).
  - Engineering, security, design, and development lead.
  - Generic soft skills nearly vanish: *communication* drops from 38% to 2%, and *education* from 37% to 1%.
  - Cloud and security skills are over-represented: *aws* 5% vs 2.6%, *security* 16% vs 14%.
- **Association rules.** At 0.05 support there are no rules. At 0.02 there is one: *compliance → security* (lift 2.65). **None of the top-20 training rules reappear.** Most training itemsets occur in under 1% of live postings: the snippets are too short and the sample is IT-only.
- **Prediction.** The Stage 5 model gets **59% accuracy (macro F1 0.48)**. That is below the 62% "always High" baseline and well below its 0.729 test accuracy. On the 57 advertised salaries it gets 51%. The model is missing experience level, company size, and industry, and sees about a sixth of the usual skills. Most "actual" tiers are Adzuna model estimates. Mid jobs are mostly predicted High (175 of 359).
- **Conclusion.** The training finding that technical work pays more holds on current postings: IT jobs sit mostly in the High tier. But this live source can't validate the rules or the classifier. Doing that needs full descriptions, advertised (not estimated) salaries, and the experience level, company size, and industry fields that the free Adzuna search does not return.